# Phase 4 — Feature Engineering & Naive Baseline
**Project:** Jijiga Flood & Drought Risk Prediction  
**Week 12** | Input: `era5_labeled.parquet` → Output: `feature_matrix.parquet`

This notebook builds the 40-feature lag matrix that XGBoost trains on, defines the
temporal train/test split, computes the naive persistence baseline that models must beat,
and performs an independent data-leakage review.

**Feature matrix:** 5 lag offsets × 8 variables = **40 features per row**  
Lag offsets: 1, 3, 7, 14, 365 days  
Variables: SPEI-6, API-92, SMI-FC, total_ro, tp, t2m, e, pev

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

DATA_PATH = '../src/data/processed/era5_labeled.parquet'
OUT_PATH  = '../src/data/processed/feature_matrix.parquet'

FEATURE_VARS = ['spei_6', 'api_92', 'smi_fc', 'total_ro', 'tp', 't2m', 'e', 'pev']
LAGS         = [1, 3, 7, 14, 365]
HORIZONS     = [1, 3, 7, 14]
LABEL_MAP    = {0: 'Low', 1: 'Moderate', 2: 'Elevated', 3: 'High', 4: 'Extreme'}

---
## P1 — Build Lag Feature Matrix

In [ ]:
df = pd.read_parquet(DATA_PATH)
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time').reset_index(drop=True)

print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Date range: {df.time.min().date()} → {df.time.max().date()}')

# Verify Phase 3 outputs present
required = ['spei_6', 'spei_12', 'api_92', 'smi_fc', 'total_ro',
            'drought_risk', 'flood_risk'] + \
           [f'drought_risk_t_plus_{n}' for n in HORIZONS] + \
           [f'flood_risk_t_plus_{n}'   for n in HORIZONS]
missing = [c for c in required if c not in df.columns]
print(f'Required columns present: {len(missing) == 0}  (missing: {missing})')

In [ ]:
# ── Create lag columns ─────────────────────────────────────────────────────────
feature_cols = []
for var in FEATURE_VARS:
    for lag in LAGS:
        col_name = f'{var}_lag_{lag}'
        df[col_name] = df[var].shift(lag)
        feature_cols.append(col_name)

print(f'Created {len(feature_cols)} lag features')
print(f'First 10: {feature_cols[:10]}')

# Drop first 365 rows — they have NaN for lag_365 features
n_before = len(df)
df = df.dropna(subset=[f'{v}_lag_365' for v in FEATURE_VARS]).reset_index(drop=True)
print(f'\nRows dropped (lag_365 NaN): {n_before - len(df)}')
print(f'Remaining rows: {len(df)}  (expected ≈ {n_before - 365})')
print(f'Date range after drop: {df.time.min().date()} → {df.time.max().date()}')

In [ ]:
# ── Visualise lag feature correlations with targets ────────────────────────────
target_cols = ([f'drought_risk_t_plus_{n}' for n in HORIZONS] +
               [f'flood_risk_t_plus_{n}'   for n in HORIZONS])

# Compute Spearman correlation of each feature vs each target
corr_matrix = df[feature_cols + target_cols].corr(method='spearman')[target_cols].loc[feature_cols]

fig, ax = plt.subplots(figsize=(10, 14))
sns.heatmap(corr_matrix, cmap='RdBu_r', center=0, vmin=-0.7, vmax=0.7,
            yticklabels=True, xticklabels=True, ax=ax, linewidths=0.1)
ax.set_title('Spearman Correlation: Lag Features vs Forecast Targets', fontsize=11)
ax.tick_params(labelsize=7)
plt.tight_layout()
plt.savefig('phase4_feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nStrongest positive correlations with drought_risk_t_plus_1:')
print(corr_matrix['drought_risk_t_plus_1'].abs().nlargest(5))

---
## P2 — Train / Test Split & Naive Persistence Baseline

In [ ]:
# ── Strict temporal split — NEVER shuffle ────────────────────────────────────
train_mask = df['time'].dt.year <= 2022
test_mask  = df['time'].dt.year >= 2023

print('Train set:')
print(f'  Rows : {train_mask.sum()}')
print(f'  Range: {df.loc[train_mask, "time"].min().date()} → {df.loc[train_mask, "time"].max().date()}')
print('Test set:')
print(f'  Rows : {test_mask.sum()}')
print(f'  Range: {df.loc[test_mask, "time"].min().date()} → {df.loc[test_mask, "time"].max().date()}')

# Walk-forward CV fold definitions (used in Phase 5)
WF_FOLDS = [
    {'train': (2001, 2014), 'val': (2015, 2016)},
    {'train': (2001, 2016), 'val': (2017, 2018)},
    {'train': (2001, 2018), 'val': (2019, 2020)},
]
print('\nWalk-forward CV folds (within training set):')
for i, f in enumerate(WF_FOLDS):
    print(f'  Fold {i+1}: train {f["train"][0]}–{f["train"][1]} | validate {f["val"][0]}–{f["val"][1]}')

In [ ]:
# ── Naive persistence baseline: predict t+n risk = current risk ───────────────
print('Naive Persistence Baseline — Weighted F1 on Test Set (2023–2025)')
print(f'{"Task":<30} {"Baseline F1":>12}')
print('-' * 44)

baseline_scores = {}
for hazard in ['drought', 'flood']:
    for n in HORIZONS:
        target  = f'{hazard}_risk_t_plus_{n}'
        current = f'{hazard}_risk'
        test_df = df[test_mask].dropna(subset=[target, current])
        if len(test_df) == 0:
            continue
        y_true = test_df[target].astype(int)
        y_pred = test_df[current].astype(int)
        f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        baseline_scores[target] = round(f1, 4)
        print(f'  {target:<30} {f1:>12.4f}')

# Visualise baseline degradation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, hazard in zip(axes, ['drought', 'flood']):
    scores = [baseline_scores.get(f'{hazard}_risk_t_plus_{n}', 0) for n in HORIZONS]
    ax.plot(HORIZONS, scores, 'o--', color='gray', ms=8, lw=2)
    for h, s in zip(HORIZONS, scores):
        ax.annotate(f'{s:.3f}', (h, s), textcoords='offset points', xytext=(0, 8), fontsize=9)
    ax.set_xlabel('Forecast Horizon (days)')
    ax.set_ylabel('Weighted F1')
    ax.set_title(f'{hazard.title()} — Naive Persistence Baseline')
    ax.set_xticks(HORIZONS)
    ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('phase4_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

---
## P3 — Independent Leakage Review

In [ ]:
# ── Data leakage review ────────────────────────────────────────────────────────
feat_set   = set(feature_cols)
target_set = set(target_cols)
risk_set   = {'drought_risk', 'flood_risk'}

print('=== Leakage Review ===')

# Check 1: No target column appears in feature list
leak1 = feat_set & target_set
print(f'[1] Target columns in feature list: {leak1 or "NONE ✓"}')

# Check 2: Original risk labels not in feature list (they are current-day labels)
leak2 = feat_set & risk_set
print(f'[2] Current-day risk labels in features: {leak2 or "NONE ✓"}')

# Check 3: Verify lag_1 on date D == value on date D-1
print('[3] Lag direction spot-checks (spei_6_lag_1):')
np.random.seed(42)
for idx in np.random.randint(365, len(df) - 15, 5):
    date   = df.loc[idx,     'time']
    lag1   = df.loc[idx,     'spei_6_lag_1']
    prev   = df.loc[idx - 1, 'spei_6']
    match  = np.isclose(lag1, prev, equal_nan=True)
    print(f'    {date.date()}  lag1={lag1:.4f}  prev_day={prev:.4f}  OK={match}')

# Check 4: Target shift direction (drought_risk_t+7 on D == drought_risk on D+7)
print('[4] Target shift direction spot-checks (drought_risk_t+7):')
for idx in np.random.randint(365, len(df) - 15, 5):
    date    = df.loc[idx,     'time']
    t7      = df.loc[idx,     'drought_risk_t_plus_7']
    future  = df.loc[idx + 7, 'drought_risk']
    match   = (t7 == future) or (pd.isna(t7) and pd.isna(future))
    print(f'    {date.date()}  t+7={t7}  actual@+7days={future}  OK={match}')

# Check 5: No test rows leak into any walk-forward fold
print('[5] Test rows in walk-forward folds:')
for f in WF_FOLDS:
    tr_mask = (df['time'].dt.year >= f['train'][0]) & (df['time'].dt.year <= f['train'][1])
    va_mask = (df['time'].dt.year >= f['val'][0])   & (df['time'].dt.year <= f['val'][1])
    test_in_train = (tr_mask & (df['time'].dt.year >= 2023)).sum()
    test_in_val   = (va_mask & (df['time'].dt.year >= 2023)).sum()
    print(f'    Fold train={f["train"]} val={f["val"]}: test rows in train={test_in_train} val={test_in_val} ✓')

---
## P4 — Data Preparation Summary

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
print('=' * 65)
print('DATA PREPARATION SUMMARY — Jijiga Flood & Drought Risk Pipeline')
print('=' * 65)
print()
print('Source data')
print('  ERA5 6-hourly → daily aggregation, 2000-01-01 to 2025-12-31')
print('  Grid point   : 42.75°E, 9.25°N  (nearest-neighbour selection)')
print('  Aggregation  : tp/e/pev/ssro/sro/lsp sum | t2m/swvl1/swvl2 mean')
print()
print('Indices')
print(f'  SPEI-6/12    : monthly log-logistic fit, ref period 2000–{CLIM_END}, ffill to daily')
print('  API-92       : k=0.92 exponential decay, init=0 on 2000-01-01')
print('  SMI-FC       : (swvl1+swvl2)/(0.323+0.323), clamped [0,1]')
print('  total_ro     : ssro + sro [m/day]')
print()
print('Risk labels')
print('  Drought: SPEI-6 McKee thresholds; modifier (+1 level) if')
print('           base∈{1,2,3} AND (API<p25_train OR SMI<p25_train)')
print('  Flood  : composite 0.40·norm_API + 0.35·norm_SMI + 0.25·norm_ro')
print('           percentile thresholds p65/p80/p90/p97 from training data')
print()
print('Feature matrix')
print(f'  Variables    : {FEATURE_VARS}')
print(f'  Lag offsets  : {LAGS} days')
print(f'  Total features: {len(feature_cols)}')
print(f'  Rows (after lag-365 drop): {len(df)}')
print()
print('Splits')
print(f'  Train: 2001-01-01 to 2022-12-31  ({train_mask.sum()} rows)')
print(f'  Test : 2023-01-01 to 2025-12-31  ({test_mask.sum()} rows)')
print('  Validation: 3-fold walk-forward within training')
print()
print('Naive persistence baseline (weighted F1 on test set):')
for k, v in baseline_scores.items():
    print(f'  {k:<30}: {v:.4f}')

In [ ]:
# ── Save feature matrix ───────────────────────────────────────────────────────
keep_cols = (['time', 'drought_risk', 'flood_risk'] +
             target_cols + feature_cols)
df_feat = df[keep_cols].copy()

df_feat.to_parquet(OUT_PATH, index=False)
size_mb = pd.io.common.get_handle(OUT_PATH, 'rb').handle.seek(0, 2) / 1e6 if False else 0
print(f'Saved: {OUT_PATH}')
print(f'Shape: {df_feat.shape}')
print(f'Columns: time, drought_risk, flood_risk, 8 targets, {len(feature_cols)} lag features')

# Also export baseline scores for Phase 5 comparison
pd.Series(baseline_scores, name='baseline_f1').to_csv('../src/data/processed/baseline_scores.csv')
print('Baseline scores exported to baseline_scores.csv')